# 🧠 NS-ARC Deep Transformer Scaling Experiment

**Three model variants with Attention Residual Transformer backbones at 32, 64, 128 depth — applied to every module (Encoder, World Model, Planner).**

---
| Model | Encoder Depth | World Model Depth | Planner Sims |
|-------|--------------|------------------|--------------|
| `NSARC-32`  | 4  | 32  | 25  |
| `NSARC-64`  | 8  | 64  | 50  |
| `NSARC-128` | 12 | 128 | 100 |

Each module of each model is saved separately for per-module depth analysis.

## Cell 0 — Bootstrap (Kaggle / Colab)

In [58]:
import sys
import os
import subprocess

def run(cmd):
    print(f"Executing: {cmd}")
    subprocess.run(cmd, shell=True, check=True)

# ─── 1. KAGGLE PATH SETUP ───────────────────────────────────────────
# If we are in Kaggle, the repo gets cloned to a subfolder. 
# We MUST add this subfolder to the very front of the Python path.
KAGGLE_REPO_PATH = '/kaggle/working/Model-Jepa'

if os.path.exists(KAGGLE_REPO_PATH):
    print("✅ Kaggle Repository Structure Detected.")
    
    # NEW: Pull the latest changes we just pushed (the arc_data rename)
    run(f"cd {KAGGLE_REPO_PATH} && git pull origin main")
    
    if KAGGLE_REPO_PATH not in sys.path:
        sys.path.insert(0, KAGGLE_REPO_PATH)
        print(f"✅ Added {KAGGLE_REPO_PATH} to sys.path")
else:
    # Fallback if running locally
    sys.path.insert(0, os.path.abspath('.'))

# ─── 2. INSTALL DEPENDENCIES ────────────────────────────────────────
run('pip install -q wandb umap-learn scikit-learn tqdm seaborn')

# ─── 3. CLONE EXTERNAL DATASET ──────────────────────────────────────
# Notice we are now using arc_data instead of data!
TARGET_DATA_DIR = f"{KAGGLE_REPO_PATH}/arc_data/re-arc" if os.path.exists(KAGGLE_REPO_PATH) else "arc_data/re-arc"

if not os.path.isdir(TARGET_DATA_DIR):
    print("Cloning Re-ARC dataset...")
    run(f'git clone --quiet https://github.com/michaelhodel/re-arc {TARGET_DATA_DIR}')
    run(f'cd {TARGET_DATA_DIR} && unzip -q re_arc.zip && unzip -q arc_original.zip')
    print('✅ Re-ARC dataset ready!')

print('🚀 Bootstrap complete.')


✅ Kaggle Repository Structure Detected.
Executing: cd /kaggle/working/Model-Jepa && git pull origin main


From https://github.com/Avadhak47/Model-Jepa
 * branch            main       -> FETCH_HEAD
   3c24f42..11591a8  main       -> origin/main


Updating 3c24f42..11591a8
Fast-forward
 analysis/__init__.py                  |   1 +
 arc_data/__init__.py                  |   1 +
 {data => arc_data}/arc_dataset.py     |   0
 {data => arc_data}/re-arc             |   0
 {data => arc_data}/rearc_dataset.py   |   0
 {data => arc_data}/terrain_dataset.py |   0
 core/__init__.py                      |   1 +
 envs/__init__.py                      |   1 +
 experiment.ipynb                      | 773 +++++++++-------------------------
 kaggle_setup.sh                       |  16 +-
 main.py                               |  12 +-
 modules/__init__.py                   |   1 +
 modules/world_models.py               |  11 +-
 tests/test_modules.py                 |   2 +-
 training/__init__.py                  |   1 +
 15 files changed, 242 insertions(+), 578 deletions(-)
 create mode 100644 analysis/__init__.py
 create mode 100644 arc_data/__init__.py
 rename {data => arc_data}/arc_dataset.py (100%)
 rename {data => arc_data}/re-arc (100%)


In [59]:
import sys, os
sys.path.insert(0, os.path.abspath('.'))

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib
matplotlib.use('Agg')  
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from collections import defaultdict
import time, json, pathlib

# ── Detect best device ──────────────────────────────────────────────────
if torch.cuda.is_available():
    DEVICE = 'cuda'
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    DEVICE = 'mps'
else:
    DEVICE = 'cpu'

print(f'Using device: {DEVICE}')
if DEVICE == 'cuda': print(torch.cuda.get_device_name(0))
torch.manual_seed(42)
np.random.seed(42)

Using device: cuda
Tesla T4


## Cell W1 — Weights & Biases Setup

In [60]:
import wandb

# ── Paste your key from https://wandb.ai/authorize ───────────────────────
WANDB_KEY = os.environ.get('WANDB_API_KEY', '')

if WANDB_KEY:
    wandb.login(key=WANDB_KEY)
else:
    wandb.login()

WANDB_PROJECT = 'NS-ARC-Scaling'
WANDB_ENTITY  = None
print(f'W&B project  : {WANDB_PROJECT}')

W&B project  : NS-ARC-Scaling


## 1. Configuration: Three Depth Profiles

In [61]:
BASE_CFG = dict(
    latent_dim   = 128,
    action_dim   = 10,
    hidden_dim   = 256,
    nhead        = 8,
    in_channels  = 1,
    device       = DEVICE,
    use_attn_res = True,
    ent_coef     = 0.01,
    gamma        = 0.99,
    lam          = 0.95,
    vocab_size   = 50,
    pb_c_init    = 1.25,
    pb_c_base    = 19652,
    curiosity_scale = 0.5,
)

PROFILES = {
    'NSARC-32': dict(**BASE_CFG,
        num_layers = 32,        # World Model depth
        enc_depth  = 4,         # Transformer Encoder depth
        num_simulations = 25,   # MCTS planner simulations
    ),
    'NSARC-64': dict(**BASE_CFG,
        num_layers = 64,
        enc_depth  = 8,
        num_simulations = 50,
    ),
    'NSARC-128': dict(**BASE_CFG,
        num_layers = 128,
        enc_depth  = 12,
        num_simulations = 100,
    ),
}

SAVE_DIR = pathlib.Path('./checkpoints')
SAVE_DIR.mkdir(exist_ok=True)
for name in PROFILES:
    (SAVE_DIR / name).mkdir(exist_ok=True)

print('Profiles ready')

Profiles ready


## 2. Environment & Dataset Setup

In [64]:

from arc_data.rearc_dataset import ReARCDataset
from envs.arc_env import ARCEnvironment

REARC_PATH = os.environ.get('REARC_DATA_PATH', 'data/re-arc')
dataset    = ReARCDataset('/kaggle/working/data/re-arc/re_arc/tasks', max_pairs_per_task=100)

env_cfg = {**BASE_CFG, 'rearc_path': REARC_PATH}
env     = ARCEnvironment(env_cfg)

print(f'Dataset pairs   : {len(dataset)}')
print(f'Sample reward   : {env.step(env.sample_action())[1]:.4f}')

ReARCDataset: loaded 40,000 grid pairs from 400 tasks in '/kaggle/working/data/re-arc/re_arc/tasks'
Dataset pairs   : 40000
Sample reward   : 2.0000


## 3. Build the Three Models

In [65]:
from modules.encoders      import TransformerEncoder
from modules.decoders      import TransformerDecoder
from modules.world_models  import TransformerWorldModel
from modules.policies      import PPOPolicy
from modules.planners      import MCTSPlanner
from modules.curiosity     import RNDCuriosity

class DeepTransformerEncoder(TransformerEncoder):
    def __init__(self, config):
        cfg = dict(config)
        cfg['_enc_depth'] = cfg.get('enc_depth', 4)
        super().__init__(cfg)
        embed_dim = cfg.get('hidden_dim', 256)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=8, dim_feedforward=embed_dim*4,
            batch_first=True, norm_first=True
        )
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=cfg['_enc_depth'])
        self.to(self.device)

def build_model(name: str, cfg: dict) -> dict:
    return dict(
        encoder     = DeepTransformerEncoder(cfg),
        decoder     = TransformerDecoder(cfg),
        world_model = TransformerWorldModel(cfg),
        policy      = PPOPolicy(cfg),
        planner     = MCTSPlanner(cfg),
        curiosity   = RNDCuriosity(cfg),
    )

MODELS = {name: build_model(name, cfg) for name, cfg in PROFILES.items()}
print('Models built')

/kaggle/working/Model-Jepa/Model-Jepa/modules/encoders.py:77: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=12)
/tmp/ipykernel_80/3350954026.py:18: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(enc_layer, num_layers=cfg['_enc_depth'])
/kaggle/working/Model-Jepa/Model-Jepa/modules/decoders.py:26: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(dec_layer, num_layers=2)


Models built


## 4. Metric Tracker

In [66]:
class MetricTracker:
    def __init__(self, model_name: str):
        self.model_name = model_name
        self.history = defaultdict(list)

    def log(self, phase: str, step: int, metrics: dict):
        for k, v in metrics.items():
            val = v.item() if hasattr(v, 'item') else float(v)
            self.history[f'{phase}/{k}'].append((step, val))

    def get(self, key: str): return [v for _, v in self.history.get(key, [])]
    def steps(self, key: str): return [s for s, _ in self.history.get(key, [])]
    def save(self, path): 
        with open(path, 'w') as f: json.dump({k: list(v) for k, v in self.history.items()}, f)

TRACKERS = {name: MetricTracker(name) for name in PROFILES}

## 5. Training Epoch Logic (Step-Synchronized with T=7 Unrolls)

In [67]:
def train_phase(phase, modules, cfg, tracker, current_step, wb_run, n_batches, batch_size):
    # Asymmetric Optimizers for Safety
    if phase == 'ae':
        # Phase 1: Autoencoder (Reconstruction + RiJEPA EBC)
        opt = torch.optim.AdamW(list(modules['encoder'].parameters()) + list(modules['decoder'].parameters()), lr=1e-3)
    elif phase == 'wm':
        # Phase 2: World Model (LeWM technique) -> low LR for encoder
        opt = torch.optim.AdamW([
            {'params': modules['world_model'].parameters(), 'lr': 1e-4},
            {'params': modules['encoder'].parameters(), 'lr': 1e-5}
        ])
    elif phase in ['pol', 'cur']:
        # Phase 3: Downstream (Encoder frozen)
        opt = torch.optim.AdamW(list(modules['policy'].parameters()) + list(modules['curiosity'].parameters()), lr=3e-4)
    else: 
        # Phase 4: Safe End-to-End
        opt = torch.optim.AdamW([
            {'params': modules['encoder'].parameters(), 'lr': 1e-6},
            {'params': modules['world_model'].parameters(), 'lr': 1e-6},
            {'params': modules['policy'].parameters(), 'lr': 1e-4},
            {'params': modules['curiosity'].parameters(), 'lr': 1e-4}
        ])

    T_UNROLL = 7

    for b in range(n_batches):
        batch = dataset.sample(batch_size)
        states = batch['state'].to(cfg['device'])
        
        if phase == 'ae':
            z_dict = modules['encoder']({'state': states})
            out = modules['decoder']({'latent': z_dict['latent'], 'state': states})
            loss_dict = modules['decoder'].loss({'state': states}, out)
        
        elif phase == 'wm':
            # Autoregressive Latent Rollouts (T = 7)
            z_start = modules['encoder']({'state': states})['latent'].detach()
            z_seq = z_start.unsqueeze(1) # [B, 1, 128]
            a_seq = torch.randn(batch_size, 1, cfg['action_dim'], device=cfg['device'])
            
            for t in range(T_UNROLL - 1):
                out = modules['world_model']({'latent': z_seq, 'action': a_seq})
                next_z = out['next_latent'][:, -1:] # Predict next token
                z_seq = torch.cat([z_seq, next_z], dim=1) # Append to sequence
                next_a = torch.randn(batch_size, 1, cfg['action_dim'], device=cfg['device'])
                a_seq = torch.cat([a_seq, next_a], dim=1)
                
            # Mock Target for final predicted state
            final_pred_z = out['next_latent'][:, -1]
            final_pred_r = out['predicted_reward'][:, -1]
            
            # We pass the final step predictions for MSE against a target
            loss_dict = modules['world_model'].loss(
                {'target_latent': torch.randn_like(final_pred_z), 'target_reward': torch.zeros(batch_size, device=cfg['device'])}, 
                {'next_latent': final_pred_z, 'predicted_reward': final_pred_r, 'attention_entropy': out['attention_entropy']}
            )
        
        elif phase == 'pol':
            z = modules['encoder']({'state': states})['latent'].detach()
            out = modules['policy']({'latent': z})
            loss_dict = {'loss': -out['entropy'] + 0.01 * out['value'].pow(2).mean()}
            
        elif phase == 'cur':
            z = modules['encoder']({'state': states})['latent'].detach()
            out = modules['curiosity']({'latent': z})
            loss_dict = modules['curiosity'].loss({}, out)
            
        else: # e2e
            z_start = modules['encoder']({'state': states})['latent']
            z_seq = z_start.unsqueeze(1)
            a_seq = torch.randn(batch_size, 1, cfg['action_dim'], device=cfg['device'])
            
            for t in range(T_UNROLL - 1):
                wm_out = modules['world_model']({'latent': z_seq, 'action': a_seq})
                next_z = wm_out['next_latent'][:, -1:]
                z_seq = torch.cat([z_seq, next_z], dim=1)
                next_a = torch.randn(batch_size, 1, cfg['action_dim'], device=cfg['device'])
                a_seq = torch.cat([a_seq, next_a], dim=1)
                
            pol_out = modules['policy']({'latent': z_seq[:, -1]})
            loss_dict = {'loss': pol_out['entropy'].mean() + (z_seq.mean() * 0.0)}

        opt.zero_grad()
        loss_dict['loss'].backward()
        torch.nn.utils.clip_grad_norm_([p for m in modules.values() for p in m.parameters()], max_norm=1.0)
        opt.step()
        
        step = current_step + b
        tracker.log(phase, step, loss_dict)
        if wb_run: 
            wb_run.log({f'{phase}/{k}': (v.item() if hasattr(v,'item') else v) for k,v in loss_dict.items()}, step=step)

    return current_step + n_batches

def save_and_upload(model_name: str, modules: dict, wb_run=None):
    """Save modules and upload as W&B artifacts while the run is active."""
    for mod_name, mod in modules.items():
        path = SAVE_DIR / model_name / f'{mod_name}.pt'
        torch.save(mod.state_dict(), path)
        if wb_run:
            art = wandb.Artifact(f'{model_name}_{mod_name}', type='model')
            art.add_file(str(path))
            wb_run.log_artifact(art)
            
print('Handlers ready')

Handlers ready


## 6. Training Loop (Curriculum Gated)

In [ ]:
N_EPOCHS = 200  # Expanded epochs to allow AE convergence before moving phases
BATCHES_PER_PHASE = 1000
BATCH_SIZE = 16
AE_LOSS_THRESHOLD = 0.3  # Max allowable reconstruction loss before downstream training starts

for model_name, modules in MODELS.items():
    cfg = PROFILES[model_name]
    wb_run = wandb.init(project=WANDB_PROJECT, name=model_name, config=cfg, reinit=True)
    
    print(f"\n{'='*55}\n Training {model_name} | W&B: {wb_run.url}\n{'='*55}")
    
    current_step = 0
    for epoch in range(N_EPOCHS):
        # 1. Phase 1: RiJEPA EBC Autoencoding
        current_step = train_phase('ae',  modules, cfg, TRACKERS[model_name], current_step, wb_run, BATCHES_PER_PHASE, BATCH_SIZE)
        
        # -- CURRICULUM GATE: Check Encoder Stability --
        recent_ae_losses = TRACKERS[model_name].get('ae/recon_loss')[-BATCHES_PER_PHASE:]
        mean_ae_loss = sum(recent_ae_losses) / len(recent_ae_losses) if recent_ae_losses else float('inf')
        
        if mean_ae_loss > AE_LOSS_THRESHOLD:
            print(f' Epoch {epoch+1:>2}/{N_EPOCHS} | AE Recon Loss ({mean_ae_loss:.4f}) > {AE_LOSS_THRESHOLD}. Encoder unstable, skipping downstream phases.')
            continue  # Skip to next epoch, train AE more.
            
        print(f' Epoch {epoch+1:>2}/{N_EPOCHS} | AE Recon Loss ({mean_ae_loss:.4f}) <= {AE_LOSS_THRESHOLD}. Training downstream physics!')
        # ----------------------------------------------
        
        # 2. Phase 2: Deep Transformer World Model (LeWM technique)
        current_step = train_phase('wm',  modules, cfg, TRACKERS[model_name], current_step, wb_run, BATCHES_PER_PHASE, BATCH_SIZE)
        
        # 3. Phase 3: Downstream Decision Making
        current_step = train_phase('pol', modules, cfg, TRACKERS[model_name], current_step, wb_run, BATCHES_PER_PHASE, BATCH_SIZE)
        current_step = train_phase('cur', modules, cfg, TRACKERS[model_name], current_step, wb_run, BATCHES_PER_PHASE, BATCH_SIZE)
        
        # 4. Phase 4: Joint Safe Tuning
        current_step = train_phase('e2e', modules, cfg, TRACKERS[model_name], current_step, wb_run, BATCHES_PER_PHASE, BATCH_SIZE)
        
    
    # ⚠️ CRITICAL: Save and upload artifacts BEFORE finishing the run
    save_and_upload(model_name, modules, wb_run=wb_run)
    
    wb_run.finish()
    
print('\nAll models trained!')

ae/color_ebc_loss,▃▂▃█▁▁▃▁▂▃▂▁▂▃▂▂▂▁▅▁▁▂▄▁▂▂▁▁▁▂▄▃▁▁▂▁▁▄▁▃
ae/loss,█▆▅▆▂▅▂▅▄▅▆▄▃▅▃▃▃▆▅▃█▃▇▄▄▅▃▆█▇▆▅▃▇▇▁▅▂▃▅
ae/recon_loss,█▄▄▁▂▆▄▆▄▄▆▂▃▁▃▁▄▁▃▂▄▃▂▃▇▄▃▆▃▂▂▃▇▆▄▄▃▅▅▃
ae/color_ebc_loss,0.00052
ae/loss,0.42664
ae/recon_loss,0.42638



 Training NSARC-32 | W&B: https://wandb.ai/avadheshkumarajay-indian-institute-of-technology-delhi/NS-ARC-Scaling/runs/jpwrdi45
 Epoch  1/200 | AE Recon Loss (0.5634) > 0.2. Encoder unstable, skipping downstream phases.
 Epoch  2/200 | AE Recon Loss (0.5670) > 0.2. Encoder unstable, skipping downstream phases.
 Epoch  3/200 | AE Recon Loss (0.5664) > 0.2. Encoder unstable, skipping downstream phases.
 Epoch  4/200 | AE Recon Loss (0.5750) > 0.2. Encoder unstable, skipping downstream phases.
 Epoch  5/200 | AE Recon Loss (0.5703) > 0.2. Encoder unstable, skipping downstream phases.
 Epoch  6/200 | AE Recon Loss (0.5630) > 0.2. Encoder unstable, skipping downstream phases.
 Epoch  7/200 | AE Recon Loss (0.5650) > 0.2. Encoder unstable, skipping downstream phases.
 Epoch  8/200 | AE Recon Loss (0.5627) > 0.2. Encoder unstable, skipping downstream phases.
 Epoch  9/200 | AE Recon Loss (0.5378) > 0.2. Encoder unstable, skipping downstream phases.
 Epoch 10/200 | AE Recon Loss (0.5631) > 0.2

ae/color_ebc_loss,▂▁▂▃▁▁▁▂▂▂▂▂▁▁▂▂▁▂▃▂▁▁▃▁▂▂▁▁▁█▁▁▁▁▁▂▁▁▂▂
ae/loss,▄▃▃▄▃▃▅▅▅▃█▅▂▂▂▂▄▅▂▄▃▂▆▃▂▃▁▂▁▂▃▃▂▃▂▃▁▁▁▃
ae/recon_loss,▃▅▆▁▄▅▃▄▄▁▆▆▂▄█▂▆▅▆▅▄▁▃▂▅▅▃▁▄▂▃▁▄▂█▃▃▂▄▄
ae/color_ebc_loss,0.00022
ae/loss,0.61137
ae/recon_loss,0.61126



 Training NSARC-64 | W&B: https://wandb.ai/avadheshkumarajay-indian-institute-of-technology-delhi/NS-ARC-Scaling/runs/15rr4ro8
 Epoch  1/200 | AE Recon Loss (0.5890) > 0.2. Encoder unstable, skipping downstream phases.
 Epoch  2/200 | AE Recon Loss (0.5817) > 0.2. Encoder unstable, skipping downstream phases.
 Epoch  3/200 | AE Recon Loss (0.5955) > 0.2. Encoder unstable, skipping downstream phases.
 Epoch  4/200 | AE Recon Loss (0.5909) > 0.2. Encoder unstable, skipping downstream phases.
 Epoch  5/200 | AE Recon Loss (0.5877) > 0.2. Encoder unstable, skipping downstream phases.
 Epoch  6/200 | AE Recon Loss (0.5858) > 0.2. Encoder unstable, skipping downstream phases.
 Epoch  7/200 | AE Recon Loss (0.5900) > 0.2. Encoder unstable, skipping downstream phases.
 Epoch  8/200 | AE Recon Loss (0.5786) > 0.2. Encoder unstable, skipping downstream phases.
 Epoch  9/200 | AE Recon Loss (0.5901) > 0.2. Encoder unstable, skipping downstream phases.
 Epoch 10/200 | AE Recon Loss (0.5650) > 0.2

In [ ]:
# Note: Replace 'NSARC-32' with whichever model checkpoint folder you want to evaluate
!pip install -q plotly
!python main.py --mode comprehensive_eval --profile NSARC-32

In [ ]:
!python main.py --mode comprehensive_eval --profile NSARC-64

In [ ]:
!python main.py --mode comprehensive_eval --profile NSARC-128

## 7. Analysis & Plotting (dashboard)

In [71]:
import matplotlib.pyplot as plt
COLORS = {'NSARC-32': '#4C9BE8', 'NSARC-64': '#E85D4A', 'NSARC-128': '#5CB85C'}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for name, tracker in TRACKERS.items():
    wm_loss = tracker.get('wm/z_loss')
    steps = tracker.steps('wm/z_loss')
    if wm_loss: axes[0].plot(steps, wm_loss, label=f'{name} WM_Loss', color=COLORS[name])
    
axes[0].set_title('World Model Latent Loss (MSE)'); axes[0].legend(); axes[0].grid(True)
plt.tight_layout(); plt.show()

/tmp/ipykernel_80/3366785100.py:10: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  axes[0].set_title('World Model Latent Loss (MSE)'); axes[0].legend(); axes[0].grid(True)
